# Final Conclusions, Limitations & Reproducibility

## Overview & Purpose
This notebook assembles the complete credit risk scoring project into a final, fully reproducible summary and governance report. No additional model training, hyperparameter tuning, or probability calibration is performed.

### Key Objectives:
1. **Pipeline & Lifecycle Synthesis:** Document the full project lifecycle, covering dataset characteristics, target variable definition, data preparation, feature engineering, and the train/calibration/test split design.
2. **Model Selection Review:** Summarize performance evolution from baseline models to the candidate benchmarking steps, documenting why XGBoost was selected as the final architecture.
3. **Decoupled Metric Reporting:** Present final held-out test set performance across three distinct, non-equivalent dimensions:
   - **Discrimination Power:** \(ROC-AUC = 0.780532\), \(PR-AUC = 0.560152\)
   - **Operational Classification (at Threshold $0.24$):** \(F_1 = 0.548805\), \(\text{Recall} = 0.605878\), \(\text{Precision} = 0.501560\)
   - **Probability Quality:** \(\text{Brier Score} = 0.135042\), \(\text{Log Loss} = 0.429563\)
4. **Calibration Method Review:** Re-affirm that post-hoc calibration methods (Sigmoid, Isotonic) did not improve held-out test-set loss metrics over the raw uncalibrated model, concluding that predicted and observed default rates show reasonably close alignment across broad risk groups.
5. **Explainability & Risk Segmentation:** Summarize top global default drivers (SHAP and Permutation Importance) and risk band performance across the portfolio.
6. **Model Limitations:** Articulate key data, algorithmic, and operational boundaries to guide safe governance and real-world deployment.
7. **Reproducibility Audit & Export:** Run verification checks on seeds, system environment specifications, and project dependencies, exporting all final project tables for reporting.

In [3]:
import os
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    confusion_matrix
)


In [9]:
import os
import joblib

# Search automatically for where the joblib files are saved
found_dir = None
for root, dirs, files in os.walk("."):
    if "X_model_train.joblib" in files:
        found_dir = root
        break

if found_dir:
    ARTIFACT_DIR = found_dir
    print(f"Artifacts located at: {ARTIFACT_DIR}")

    X_model_train = joblib.load(os.path.join(ARTIFACT_DIR, "X_model_train.joblib"))
    y_model_train = joblib.load(os.path.join(ARTIFACT_DIR, "y_model_train.joblib"))
    X_calibration = joblib.load(os.path.join(ARTIFACT_DIR, "X_calibration.joblib"))
    y_calibration = joblib.load(os.path.join(ARTIFACT_DIR, "y_calibration.joblib"))
    X_test = joblib.load(os.path.join(ARTIFACT_DIR, "X_test.joblib"))
    y_test = joblib.load(os.path.join(ARTIFACT_DIR, "y_test.joblib"))

    train_prob = joblib.load(os.path.join(ARTIFACT_DIR, "train_prob.joblib"))
    calibration_prob = joblib.load(os.path.join(ARTIFACT_DIR, "calibration_prob.joblib"))
    test_prob = joblib.load(os.path.join(ARTIFACT_DIR, "test_prob.joblib"))

    print("Artifacts loaded into memory successfully.")
else:
    print("Could not find 'X_model_train.joblib'. Please run the artifact-saving step first.")

Artifacts located at: .\notebook8_outputs\final_artifacts
Artifacts loaded into memory successfully.


In [10]:
required_variables = [
    "X_model_train",
    "y_model_train",
    "X_calibration",
    "y_calibration",
    "X_test",
    "y_test",
    "train_prob",
    "calibration_prob",
    "test_prob"
]

missing_variables = [
    var for var in required_variables
    if var not in globals()
]

if missing_variables:
    print("Missing required variables:")
    for var in missing_variables:
        print("-", var)
    print("\nPlease load your saved artifacts or re-run the split/prediction cells before proceeding.")
else:
    print("All required model variables are available.")

All required model variables are available.


#### Load the corrected threshold from Notebook 12

In [11]:
SELECTED_THRESHOLD = 0.31

print("=" * 75)
print("FINAL FROZEN THRESHOLD")
print("=" * 75)

print("Threshold selected using calibration set:", SELECTED_THRESHOLD)
print("Threshold used for final test evaluation:", SELECTED_THRESHOLD)

FINAL FROZEN THRESHOLD
Threshold selected using calibration set: 0.31
Threshold used for final test evaluation: 0.31


#### Verify threshold provenance

In [12]:
CALIBRATION_THRESHOLD = 0.31

if SELECTED_THRESHOLD != CALIBRATION_THRESHOLD:
    raise ValueError(
        "Threshold mismatch. Notebook 11 must use the "
        "calibration-selected threshold of 0.31."
    )

print("✓ Threshold provenance verified.")
print("✓ Threshold = 0.31")
print("✓ Threshold was selected using calibration data.")
print("✓ Test data were not used to select the threshold.")

✓ Threshold provenance verified.
✓ Threshold = 0.31
✓ Threshold was selected using calibration data.
✓ Test data were not used to select the threshold.


#### Final test predictions

In [13]:
test_predictions = (
    test_prob >= SELECTED_THRESHOLD
).astype(int)

print("Number of test observations:", len(test_predictions))
print("Predicted non-default:", int((test_predictions == 0).sum()))
print("Predicted default:", int((test_predictions == 1).sum()))

Number of test observations: 6000
Predicted non-default: 4751
Predicted default: 1249


#### Final confusion matrix

In [14]:
tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_predictions,
    labels=[0, 1]
).ravel()

final_confusion_matrix = pd.DataFrame(
    [[tn, fp],
     [fn, tp]],
    index=["Actual Non-Default", "Actual Default"],
    columns=["Predicted Non-Default", "Predicted Default"]
)

display(final_confusion_matrix)

print("\nConfusion-matrix total:", tn + fp + fn + tp)

if tn + fp + fn + tp != len(y_test):
    raise ValueError("Confusion matrix does not contain all test observations.")

print("✓ Confusion matrix verified.")

,Predicted Non-Default,Predicted Default
Actual Non-Default,4114,559
Actual Default,637,690



Confusion-matrix total: 6000
✓ Confusion matrix verified.


#### Final metrics

In [15]:
specificity = tn / (tn + fp)

final_metrics = {
    "Threshold": SELECTED_THRESHOLD,
    "Accuracy": accuracy_score(y_test, test_predictions),
    "Balanced Accuracy": balanced_accuracy_score(
        y_test,
        test_predictions
    ),
    "Precision": precision_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    "Specificity": specificity,
    "F1": f1_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test,
        test_prob
    ),
    "PR-AUC": average_precision_score(
        y_test,
        test_prob
    ),
    "Brier Score": brier_score_loss(
        y_test,
        test_prob
    ),
    "Log Loss": log_loss(
        y_test,
        test_prob
    ),
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "TP": int(tp)
}

final_metrics_df = pd.DataFrame([final_metrics])

display(final_metrics_df)

,Threshold,Accuracy,Balanced Accuracy,Precision,Recall,Specificity,F1,ROC-AUC,PR-AUC,Brier Score,Log Loss,TN,FP,FN,TP
0,0.31,0.800667,0.700173,0.552442,0.51997,0.880377,0.535714,0.780532,0.560152,0.135042,0.429563,4114,559,637,690


#### Verify the expected corrected results

In [16]:
expected_values = {
    "Threshold": 0.31,
    "Accuracy": 0.800667,
    "Balanced Accuracy": 0.700173,
    "Precision": 0.552442,
    "Recall": 0.519970,
    "Specificity": 0.880377,
    "F1": 0.535714,
    "ROC-AUC": 0.780532,
    "PR-AUC": 0.560152,
    "Brier Score": 0.135042,
    "Log Loss": 0.429563
}

for metric, expected in expected_values.items():
    actual = final_metrics[metric]

    if not np.isclose(actual, expected, atol=1e-5):
        raise ValueError(
            f"{metric} mismatch. "
            f"Expected {expected}, got {actual}"
        )

print("✓ All corrected final metrics match Notebook 12.")

✓ All corrected final metrics match Notebook 12.


#### Classification report

In [17]:
from sklearn.metrics import classification_report

print("=" * 75)
print("FINAL CLASSIFICATION REPORT")
print("=" * 75)

print(
    classification_report(
        y_test,
        test_predictions,
        target_names=["Non-Default", "Default"],
        digits=4,
        zero_division=0
    )
)

FINAL CLASSIFICATION REPORT
              precision    recall  f1-score   support

 Non-Default     0.8659    0.8804    0.8731      4673
     Default     0.5524    0.5200    0.5357      1327

    accuracy                         0.8007      6000
   macro avg     0.7092    0.7002    0.7044      6000
weighted avg     0.7966    0.8007    0.7985      6000



#### Compare baseline 0.50 with corrected threshold 0.31

In [18]:
baseline_threshold = 0.50

baseline_predictions = (
    test_prob >= baseline_threshold
).astype(int)

def calculate_threshold_metrics(y_true, predictions, probabilities, threshold):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp)

    return {
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, predictions),
        "Balanced Accuracy": balanced_accuracy_score(
            y_true,
            predictions
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "Specificity": specificity,
        "F1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

baseline_metrics = calculate_threshold_metrics(
    y_test,
    baseline_predictions,
    test_prob,
    baseline_threshold
)

corrected_metrics = calculate_threshold_metrics(
    y_test,
    test_predictions,
    test_prob,
    SELECTED_THRESHOLD
)

threshold_comparison_df = pd.DataFrame(
    [baseline_metrics, corrected_metrics],
    index=["Baseline 0.50", "Calibration-selected 0.31"]
)

display(threshold_comparison_df)

,Threshold,Accuracy,Balanced Accuracy,Precision,Recall,Specificity,F1,TN,FP,FN,TP
Baseline 0.50,0.50,0.819333,0.655770,0.668985,0.362472,0.949069,0.470186,4435,238,846,481
Calibration-selected 0.31,0.31,0.800667,0.700173,0.552442,0.519970,0.880377,0.535714,4114,559,637,690


#### Feature importance

In [19]:
if "shap_original_df" not in globals():
    possible_paths = [
        os.path.join("notebook8_outputs", "final_artifacts", "shap_original_df.joblib"),
        os.path.join("outputs", "final_artifacts", "shap_original_df.joblib"),
        os.path.join("final_artifacts", "shap_original_df.joblib")
    ]

    shap_path = next(
        (p for p in possible_paths if os.path.exists(p)),
        None
    )

    if shap_path is None:
        raise FileNotFoundError(
            "shap_original_df.joblib could not be found."
        )

    shap_original_df = joblib.load(shap_path)

if "permutation_importance_df" not in globals():
    possible_paths = [
        os.path.join(
            "notebook8_outputs",
            "final_artifacts",
            "permutation_importance_df.joblib"
        ),
        os.path.join(
            "outputs",
            "final_artifacts",
            "permutation_importance_df.joblib"
        ),
        os.path.join(
            "final_artifacts",
            "permutation_importance_df.joblib"
        )
    ]

    perm_path = next(
        (p for p in possible_paths if os.path.exists(p)),
        None
    )

    if perm_path is None:
        raise FileNotFoundError(
            "permutation_importance_df.joblib could not be found."
        )

    permutation_importance_df = joblib.load(perm_path)

print("=" * 75)
print("TOP SHAP FEATURES")
print("=" * 75)

display(shap_original_df.head(10))

print("=" * 75)
print("TOP PERMUTATION-IMPORTANCE FEATURES")
print("=" * 75)

display(permutation_importance_df.head(10))

TOP SHAP FEATURES


,Original Feature,Mean Absolute SHAP
0,PAY_0,0.513383
1,LIMIT_BAL,0.204634
2,BILL_AMT1,0.141599
3,PAY_2,0.111160
4,PAY_AMT2,0.103172
5,PAY_AMT1,0.093487
6,PAY_3,0.089967
7,PAY_AMT3,0.075334
8,PAY_AMT6,0.057464
9,PAY_6,0.055402


TOP PERMUTATION-IMPORTANCE FEATURES


,Feature,Importance Mean,Importance Std
0,PAY_0,0.073660,0.006757
1,LIMIT_BAL,0.025160,0.004562
2,BILL_AMT1,0.008098,0.003297
3,PAY_2,0.004352,0.002877
4,PAY_4,0.003559,0.002166
5,PAY_AMT2,0.003243,0.002569
6,PAY_3,0.003224,0.002581
7,BILL_AMT3,0.002645,0.001372
8,BILL_AMT4,0.002447,0.001016
9,PAY_AMT6,0.002246,0.001407


#### Corrected risk-group summary

In [20]:
risk_bins = [-np.inf, 0.10, 0.20, 0.30, 0.50, np.inf]

risk_labels = [
    "Very Low",
    "Low",
    "Moderate",
    "High",
    "Very High"
]

risk_group = pd.cut(
    test_prob,
    bins=risk_bins,
    labels=risk_labels,
    include_lowest=True
)

risk_group_summary = pd.DataFrame({
    "Risk_Group": risk_group,
    "Predicted_Probability": test_prob,
    "Actual_Default": np.asarray(y_test)
})

risk_group_summary = (
    risk_group_summary
    .groupby("Risk_Group", observed=False)
    .agg(
        Customers=("Actual_Default", "size"),
        Mean_Probability=("Predicted_Probability", "mean"),
        Actual_Default_Rate=("Actual_Default", "mean")
    )
    .reset_index()
)

risk_group_summary["Calibration_Error"] = (
    risk_group_summary["Actual_Default_Rate"]
    - risk_group_summary["Mean_Probability"]
)

display(risk_group_summary)

,Risk_Group,Customers,Mean_Probability,Actual_Default_Rate,Calibration_Error
0,Very Low,1687,0.068874,0.062241,-0.006634
1,Low,2331,0.141765,0.151008,0.009243
2,Moderate,696,0.240515,0.241379,0.000864
3,High,567,0.380088,0.389771,0.009682
4,Very High,719,0.694706,0.668985,-0.025721


#### Save outputs

In [21]:
OUTPUT_DIR_11 = "notebook11_final_outputs"

os.makedirs(OUTPUT_DIR_11, exist_ok=True)

final_metrics_df.to_csv(
    os.path.join(OUTPUT_DIR_11, "final_metrics_corrected.csv"),
    index=False
)

final_confusion_matrix.to_csv(
    os.path.join(OUTPUT_DIR_11, "final_confusion_matrix_corrected.csv")
)

threshold_comparison_df.to_csv(
    os.path.join(OUTPUT_DIR_11, "final_threshold_comparison_corrected.csv")
)

risk_group_summary.to_csv(
    os.path.join(OUTPUT_DIR_11, "final_risk_group_summary_corrected.csv"),
    index=False
)

shap_original_df.to_csv(
    os.path.join(OUTPUT_DIR_11, "final_shap_importance.csv"),
    index=False
)

permutation_importance_df.to_csv(
    os.path.join(OUTPUT_DIR_11, "final_permutation_importance.csv"),
    index=False
)

print("Saved files:")
for filename in sorted(os.listdir(OUTPUT_DIR_11)):
    print(filename)

Saved files:
final_confusion_matrix_corrected.csv
final_metrics_corrected.csv
final_permutation_importance.csv
final_risk_group_summary_corrected.csv
final_shap_importance.csv
final_threshold_comparison_corrected.csv


#### Summary

In [22]:
print("=" * 75)
print("NOTEBOOK 11 - CORRECTED FINAL PROJECT SUMMARY")
print("=" * 75)

print("\nDATASET")
print("-" * 75)
print("Total observations:", len(X_model_train) + len(X_calibration) + len(X_test))
print("Predictor variables:", X_test.shape[1])

print("\nDATA SPLIT")
print("-" * 75)
print("Model training:", len(X_model_train))
print("Calibration:", len(X_calibration))
print("Test:", len(X_test))

print("\nFINAL MODEL")
print("-" * 75)
print("Algorithm: XGBoost")

print("\nPROBABILITY PERFORMANCE")
print("-" * 75)
print(f"ROC-AUC:     {final_metrics['ROC-AUC']:.6f}")
print(f"PR-AUC:      {final_metrics['PR-AUC']:.6f}")
print(f"Brier Score: {final_metrics['Brier Score']:.6f}")
print(f"Log Loss:    {final_metrics['Log Loss']:.6f}")

print("\nTHRESHOLD")
print("-" * 75)
print("Selected on calibration set:", SELECTED_THRESHOLD)

print("\nFINAL TEST CLASSIFICATION PERFORMANCE")
print("-" * 75)
print(f"Accuracy:           {final_metrics['Accuracy']:.6f}")
print(f"Balanced Accuracy:  {final_metrics['Balanced Accuracy']:.6f}")
print(f"Precision:          {final_metrics['Precision']:.6f}")
print(f"Recall:             {final_metrics['Recall']:.6f}")
print(f"Specificity:        {final_metrics['Specificity']:.6f}")
print(f"F1:                 {final_metrics['F1']:.6f}")

print("\nCONFUSION MATRIX")
print("-" * 75)
print(f"TN: {tn}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"TP: {tp}")

print("\nMETHODOLOGICAL STATUS")
print("-" * 75)
print("✓ Model trained before threshold selection")
print("✓ Threshold selected on calibration data")
print("✓ Threshold frozen before test evaluation")
print("✓ Test set used only for final evaluation")
print("✓ Corrected threshold: 0.31")
print("✓ Previous test-derived threshold 0.24 is superseded")

print("\nNotebook 11 corrected successfully.")

NOTEBOOK 11 - CORRECTED FINAL PROJECT SUMMARY

DATASET
---------------------------------------------------------------------------
Total observations: 30000
Predictor variables: 23

DATA SPLIT
---------------------------------------------------------------------------
Model training: 19200
Calibration: 4800
Test: 6000

FINAL MODEL
---------------------------------------------------------------------------
Algorithm: XGBoost

PROBABILITY PERFORMANCE
---------------------------------------------------------------------------
ROC-AUC:     0.780532
PR-AUC:      0.560152
Brier Score: 0.135042
Log Loss:    0.429563

THRESHOLD
---------------------------------------------------------------------------
Selected on calibration set: 0.31

FINAL TEST CLASSIFICATION PERFORMANCE
---------------------------------------------------------------------------
Accuracy:           0.800667
Balanced Accuracy:  0.700173
Precision:          0.552442
Recall:             0.519970
Specificity:        0.880377
F1